<a href="https://colab.research.google.com/github/Mostafizur5291/0152310005101016_Mostafizur-OOP-Assignment/blob/main/Ai_Students_Performance_Prediction_Using_Mechine_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import files
import io
import pandas as pd

print("1. Prothome 'student-mat.csv' file ta select koro:")
uploaded_mat = files.upload()

print("\n2. Ekhon 'student-por.csv' file ta select koro:")
uploaded_por = files.upload()

# Auto-detect file name
mat_file = list(uploaded_mat.keys())[0]
por_file = list(uploaded_por.keys())[0]

# Load Data
mat = pd.read_csv(io.BytesIO(uploaded_mat[mat_file]), sep=";")
por = pd.read_csv(io.BytesIO(uploaded_por[por_file]), sep=";")

print("\n✅ Data Loaded Successfully!")
print("Math Data:", mat.shape)
print("Port Data:", por.shape)

1. Prothome 'student-mat.csv' file ta select koro:


Saving student-mat[1].csv to student-mat[1].csv

2. Ekhon 'student-por.csv' file ta select koro:


Saving student-por[1].csv to student-por[1].csv

✅ Data Loaded Successfully!
Math Data: (395, 33)
Port Data: (649, 33)


In [4]:
import pandas as pd

# Target build korar function
def build_target(df):
    df = df.copy()
    # G1, G2, G3 er average ber kora
    df["Avg_Score"] = df[["G1","G2","G3"]].mean(axis=1)

    # Bottom 33% student der At-Risk (0) hishebe dhora
    thresh = df["Avg_Score"].quantile(1/3)
    df["Y"] = (df["Avg_Score"] > thresh).astype(int) # 0=At-Risk, 1=On-Track
    return df, thresh

# Math ar Portuguese dutor jonnoi target build kora
mat_t, mat_thr = build_target(mat)
por_t, por_thr = build_target(por)

print("✅ Target Build Complete!\n")
print(f"Math Cutoff: {mat_thr:.2f} | At-Risk rate: {1 - mat_t['Y'].mean():.3f}")
print(f"Port Cutoff: {por_thr:.2f} | At-Risk rate: {1 - por_t['Y'].mean():.3f}")

✅ Target Build Complete!

Math Cutoff: 9.11 | At-Risk rate: 0.334
Port Cutoff: 10.33 | At-Risk rate: 0.351


In [8]:
from scipy.stats import spearmanr

TIER_AACT = ["studytime", "absences", "goout", "Dalc", "Walc"]
audit_rows = []

for f in TIER_AACT:
    # Correlation check kora
    r, p = spearmanr(mat_t[f], mat_t["Avg_Score"])

    # Direction fix kora
    constrained = (abs(r) > 0.10) and (p < 0.05)
    direction = "increase" if r > 0 else "decrease"

    audit_rows.append({
        "Feature": f,
        "r": round(r, 3),
        "p_value": round(p, 3),
        "Constrained": constrained,
        "Direction": direction if constrained else "None"
    })

audit_df = pd.DataFrame(audit_rows)
display(audit_df)

,Feature,r,p_value,Constrained,Direction
0,studytime,0.125,0.013,True,increase
1,absences,0.026,0.604,False,None
2,goout,-0.156,0.002,True,decrease
3,Dalc,-0.115,0.023,True,decrease
4,Walc,-0.097,0.053,False,None


In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from fairlearn.metrics import demographic_parity_difference, equalized_odds_difference
import pandas as pd

# 1. Features List
SENSITIVE   = ["address", "sex", "school", "Medu", "Fedu"]
IMMUTABLE   = ["age","famsize","Pstatus","Mjob","Fjob","guardian","reason","nursery","failures"]
TIER_C      = ["internet","paid","schoolsup","famsup","traveltime"]
A_MODEL     = ["studytime","absences","goout","freetime","Dalc","Walc","activities","higher","health","romantic"]

features = SENSITIVE + IMMUTABLE + TIER_C + A_MODEL

# 2. Data Prepare
df = mat_t.copy()
X = pd.get_dummies(df[features], drop_first=False).astype(float)
y = df["Y"]

# 3. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
address_test = df.loc[X_test.index, "address"]
sex_test = df.loc[X_test.index, "sex"]

# 4. Model Train
rf = RandomForestClassifier(n_estimators=300, random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

# 5. Fairness Metrics Calculate
dp_addr = demographic_parity_difference(y_test, y_pred, sensitive_features=address_test)
eo_addr = equalized_odds_difference(y_test, y_pred, sensitive_features=address_test)

dp_sex = demographic_parity_difference(y_test, y_pred, sensitive_features=sex_test)
eo_sex = equalized_odds_difference(y_test, y_pred, sensitive_features=sex_test)

# 6. Print Results
print("✅ E1 Complete!\n")
print(f"Macro-F1 Score: {f1_score(y_test, y_pred, average='macro'):.3f}\n")
print("Fairness Metrics (Target: < 0.05 ideal, > 0.10 problem):")
print(f"Address -> DP Diff: {dp_addr:.3f} | EOdds Diff: {eo_addr:.3f}")
print(f"Sex     -> DP Diff: {dp_sex:.3f} | EOdds Diff: {eo_sex:.3f}")

✅ E1 Complete!

Macro-F1 Score: 0.565

Fairness Metrics (Target: < 0.05 ideal, > 0.10 problem):
Address -> DP Diff: 0.075 | EOdds Diff: 0.143
Sex     -> DP Diff: 0.008 | EOdds Diff: 0.119


In [10]:
!pip install dice-ml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 16.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [dice-ml]


In [12]:
import numpy as np
import pandas as pd
import dice_ml
from sklearn.ensemble import RandomForestClassifier

print("E2: Recourse Cost & Structural Exhaustion shuru holo...\n")

features = ["address", "sex", "school", "Medu", "Fedu", "age", "famsize",
            "studytime", "absences", "goout", "freetime", "Dalc", "Walc",
            "internet", "paid", "schoolsup", "famsup", "traveltime", "higher", "romantic"]

df = mat_t.copy()
X = pd.get_dummies(df[features], drop_first=False).astype(float)
y = df["Y"]


rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X, y)


continuous_features = ["age", "absences", "studytime", "goout", "freetime", "Dalc", "Walc"]
d = dice_ml.Data(dataframe=pd.concat([X, y], axis=1), continuous_features=continuous_features, outcome_name="Y")
m = dice_ml.Model(model=rf, backend="sklearn")
dice = dice_ml.Dice(d, m, method="random")


at_risk_indices = df[df["Y"] == 0].head(20).index
at_risk_X = X.loc[at_risk_indices]

results = []
print("20 jon At-Risk student der jonno Counterfactual ber kora hocche...\n")

for idx, row in at_risk_X.iterrows():
    current_data = row.to_frame().T
    address_val = df.loc[idx, 'address'] # Address ekhane theke nicchi

    try:
        cf = dice.generate_counterfactuals(current_data, total_CFs=1, desired_class=1)
        cf_df = cf.cf_examples_list[0].final_cfs_df

        if cf_df is not None and len(cf_df) > 0:
            cf_row = cf_df.iloc[0]

            # Cost calculate (studytime and absences normalized change)
            study_change = abs(cf_row['studytime'] - row['studytime']) / (X['studytime'].max() - X['studytime'].min())
            abs_change = abs(cf_row['absences'] - row['absences']) / (X['absences'].max() - X['absences'].min())
            cost = (study_change + abs_change) / 2

            results.append({"Student_ID": idx, "Address": address_val, "Cost": cost, "Rescued": True})
        else:
            results.append({"Student_ID": idx, "Address": address_val, "Cost": np.nan, "Rescued": False})
    except Exception:
        results.append({"Student_ID": idx, "Address": address_val, "Cost": np.nan, "Rescued": False})

# 5. Results Analysis
results_df = pd.DataFrame(results)
urban_df = results_df[results_df['Address'] == 'U']
rural_df = results_df[results_df['Address'] == 'R']

avg_cost_urban = urban_df['Cost'].mean()
avg_cost_rural = rural_df['Cost'].mean()
exh_urban = (urban_df['Rescued'] == False).mean()
exh_rural = (rural_df['Rescued'] == False).mean()

print("✅ E2 Analysis Complete!\n")
print("--- RECOURSE COST (Average) ---")
print(f"Urban: {avg_cost_urban:.3f} | Rural: {avg_cost_rural:.3f}")
print(f"ΔCost (Rural - Urban): {avg_cost_rural - avg_cost_urban:.3f}\n")

print("--- STRUCTURAL EXHAUSTION RATE (Jader bachano somvob na) ---")
print(f"Urban: {exh_urban:.3f} | Rural: {exh_rural:.3f}")
print(f"ΔExh (Rural - Urban): {exh_rural - exh_urban:.3f}")

E2: Recourse Cost & Structural Exhaustion shuru holo...

20 jon At-Risk student der jonno Counterfactual ber kora hocche...



  0%|          | 0/1 [00:00<?, ?it/s]/usr/local/lib/python3.13/dist-packages/dice_ml/explainer_interfaces/dice_random.py:116: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1.0' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  candidate_cfs.at[k, selected_features[k][0]] = random_instances.at[k, selected_features[k][0]]
/usr/local/lib/python3.13/dist-packages/dice_ml/explainer_interfaces/dice_random.py:116: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '4.0' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  candidate_cfs.at[k, selected_features[k][0]] = random_instances.at[k, selected_features[k][0]]
/usr/local/lib/python3.13/dist-packages/dice_ml/explainer_interfaces/dice_random.py:116: FutureWarning: Setting an item of incompatible dtype is 

✅ E2 Analysis Complete!

--- RECOURSE COST (Average) ---
Urban: 0.032 | Rural: 0.039
ΔCost (Rural - Urban): 0.007

--- STRUCTURAL EXHAUSTION RATE (Jader bachano somvob na) ---
Urban: 0.000 | Rural: 0.000
ΔExh (Rural - Urban): 0.000


In [13]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

print("Full Data er upor E2 & E4 shuru holo...\n")

# 1. Data Prepare
features = ["address", "sex", "school", "Medu", "Fedu", "age", "famsize",
            "studytime", "absences", "goout", "freetime", "Dalc", "Walc",
            "internet", "paid", "schoolsup", "famsup", "traveltime", "higher", "romantic"]

df = mat_t.copy()
X = pd.get_dummies(df[features], drop_first=False).astype(float)
y = df["Y"]

# 2. Train Model
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X, y)


at_risk_indices = df[df["Y"] == 0].index
print(f"Total At-Risk Students: {len(at_risk_indices)}")

# 4. Structural Exhaustion Test & Tier-C Rescue
results = []


tier_c_cols = [c for c in X.columns if any(x in c for x in ["internet", "paid", "schoolsup", "famsup", "traveltime"])]

for idx in at_risk_indices:
    student = X.loc[idx].copy()


    student_best = student.copy()
    student_best['studytime'] = X['studytime'].max()
    student_best['absences'] = X['absences'].min()
    student_best['goout'] = X['goout'].min()
    student_best['Dalc'] = X['Dalc'].min()
    student_best['Walc'] = X['Walc'].min()


    prob_best = rf.predict_proba(student_best.values.reshape(1, -1))[0, 1]

    if prob_best < 0.5:

        rescued = False
        rescue_method = "None"


        student_rescue = student_best.copy()
        for col in tier_c_cols:
            if "internet" in col or "paid" in col or "schoolsup" in col:
                student_rescue[col] = 1.0

        prob_rescue = rf.predict_proba(student_rescue.values.reshape(1, -1))[0, 1]
        if prob_rescue >= 0.5:
            rescued = True
            rescue_method = "Tier-C (All combined)"

        results.append({
            "Student_ID": idx,
            "Address": df.loc[idx, "address"],
            "Structurally_Exhausted": True,
            "Rescued_by_Tier_C": rescued,
            "Method": rescue_method
        })
    else:
        results.append({
            "Student_ID": idx,
            "Address": df.loc[idx, "address"],
            "Structurally_Exhausted": False,
            "Rescued_by_Tier_C": False,
            "Method": "Student Effort"
        })

# 5. Analysis
results_df = pd.DataFrame(results)
exhausted_df = results_df[results_df["Structurally_Exhausted"] == True]

urban_count = len(results_df[results_df["Address"] == "U"])
rural_count = len(results_df[results_df["Address"] == "R"])

exh_urban = len(exhausted_df[exhausted_df["Address"] == "U"]) / urban_count
exh_rural = len(exhausted_df[exhausted_df["Address"] == "R"]) / rural_count

rescued_count = len(exhausted_df[exhausted_df["Rescued_by_Tier_C"] == True])
total_exhausted = len(exhausted_df)

print("\n✅ FULL E2 + E4 COMPLETE!\n")
print("--- STRUCTURAL EXHAUSTION (Pura Data) ---")
print(f"Total Exhausted: {total_exhausted} out of {len(at_risk_indices)}")
print(f"Urban Exh Rate: {exh_urban:.3f} | Rural Exh Rate: {exh_rural:.3f}")
print(f"ΔExh (Rural - Urban): {exh_rural - exh_urban:.3f}\n")

print("--- TIER-C RESCUE (Institutional Help) ---")
if total_exhausted > 0:
    print(f"Rescued by Tier-C: {rescued_count} / {total_exhausted}")
    print(f"Rescue Rate: {(rescued_count / total_exhausted) * 100:.1f}% (H3 target: >= 40%)")
else:
    print("Kono student Structural Exhaustion e nei, tai Tier-C rescue lagche na.")

Full Data er upor E2 & E4 shuru holo...

Total At-Risk Students: 132


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local


✅ FULL E2 + E4 COMPLETE!

--- STRUCTURAL EXHAUSTION (Pura Data) ---
Total Exhausted: 63 out of 132
Urban Exh Rate: 0.430 | Rural Exh Rate: 0.590
ΔExh (Rural - Urban): 0.160

--- TIER-C RESCUE (Institutional Help) ---
Rescued by Tier-C: 21 / 63
Rescue Rate: 33.3% (H3 target: >= 40%)


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [14]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from fairlearn.reductions import ExponentiatedGradient, EqualizedOdds
from sklearn.metrics import f1_score

print("E3 Pivot Analysis shuru holo...\n")

# 1. Data Prepare
features = ["address", "sex", "school", "Medu", "Fedu", "age", "famsize",
            "studytime", "absences", "goout", "freetime", "Dalc", "Walc",
            "internet", "paid", "schoolsup", "famsup", "traveltime", "higher", "romantic"]

df = mat_t.copy()
X = pd.get_dummies(df[features], drop_first=False).astype(float)
y = df["Y"]
sensitive = df["address"]

# 2. Fairlearn Model Train (Equalized Odds apply kore)
print("Fairlearn Model Train hocche (Ektu time nite pare)...")
rf = RandomForestClassifier(n_estimators=50, random_state=42)
eg = ExponentiatedGradient(rf, constraints=EqualizedOdds(), max_iter=15)
eg.fit(X, y, sensitive_features=sensitive)
print("Model Train Sesh!\n")

# 3. Fair Model diye Cost & Exhaustion check kora
at_risk_indices = df[df["Y"] == 0].index
results_fair = []

for idx in at_risk_indices:
    student = X.loc[idx].copy()
    student_best = student.copy()

    # Best effort
    student_best['studytime'] = X['studytime'].max()
    student_best['absences'] = X['absences'].min()
    student_best['goout'] = X['goout'].min()
    student_best['Dalc'] = X['Dalc'].min()
    student_best['Walc'] = X['Walc'].min()

    # Prob predict
    prob_best = eg.predict(student_best.to_frame().T)[0]

    if prob_best == 0: # Exhausted (Fair model e)
        results_fair.append({
            "Address": df.loc[idx, "address"],
            "Exhausted": True
        })
    else:
        results_fair.append({
            "Address": df.loc[idx, "address"],
            "Exhausted": False
        })

# 4. Result
res_df = pd.DataFrame(results_fair)
urban_exh = res_df[res_df["Address"] == "U"]["Exhausted"].mean()
rural_exh = res_df[res_df["Address"] == "R"]["Exhausted"].mean()

delta_exh_fair = rural_exh - urban_exh

print("✅ E3 PIVOT RESULT\n")
print("--- FAIR MODEL ER STRUCTURAL EXHAUSTION ---")
print(f"Urban Exh Rate: {urban_exh:.3f} | Rural Exh Rate: {rural_exh:.3f}")
print(f"ΔExh (Fair Model): {delta_exh_fair:.3f}\n")

print("--- THESIS ER PIVOT (E3) ---")
# Ager Baseline ΔExh chilo 0.160
baseline_delta = 0.160
change = delta_exh_fair - baseline_delta

print(f"Baseline ΔExh: {baseline_delta:.3f}")
print(f"Fair Model ΔExh: {delta_exh_fair:.3f}")
print(f"Change: {change:.3f}\n")

if abs(change) < 0.05:
    print("👉 VERDICT: FLAT! (Predictive fairness and Recourse fairness are DECOUPLED)")
    print("Aapnar thesis er primary contribution ekhanei prove hoye gelo!")
elif change < 0:
    print("👉 VERDICT: FALLS! (Fairness mitigation transfers partially)")
else:
    print("👉 VERDICT: RISES! (Trilemma)")

E3 Pivot Analysis shuru holo...

Fairlearn Model Train hocche (Ektu time nite pare)...
Model Train Sesh!

✅ E3 PIVOT RESULT

--- FAIR MODEL ER STRUCTURAL EXHAUSTION ---
Urban Exh Rate: 0.516 | Rural Exh Rate: 0.718
ΔExh (Fair Model): 0.202

--- THESIS ER PIVOT (E3) ---
Baseline ΔExh: 0.160
Fair Model ΔExh: 0.202
Change: 0.042

👉 VERDICT: FLAT! (Predictive fairness and Recourse fairness are DECOUPLED)
Aapnar thesis er primary contribution ekhanei prove hoye gelo!


In [15]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

print("E5: Portuguese Dataset Replication shuru holo...\n")

# 1. Portuguese Data Prepare
features = ["address", "sex", "school", "Medu", "Fedu", "age", "famsize",
            "studytime", "absences", "goout", "freetime", "Dalc", "Walc",
            "internet", "paid", "schoolsup", "famsup", "traveltime", "higher", "romantic"]

df_por = por_t.copy()
X_por = pd.get_dummies(df_por[features], drop_first=False).astype(float)
y_por = df_por["Y"]

# 2. Train Model
rf_por = RandomForestClassifier(n_estimators=100, random_state=42)
rf_por.fit(X_por, y_por)

# 3. Structural Exhaustion Check
at_risk_indices = df_por[df_por["Y"] == 0].index
results_por = []

for idx in at_risk_indices:
    student = X_por.loc[idx].copy()
    student_best = student.copy()

    # Best effort
    student_best['studytime'] = X_por['studytime'].max()
    student_best['absences'] = X_por['absences'].min()
    student_best['goout'] = X_por['goout'].min()
    student_best['Dalc'] = X_por['Dalc'].min()
    student_best['Walc'] = X_por['Walc'].min()

    prob_best = rf_por.predict(student_best.to_frame().T)[0]

    if prob_best == 0:
        results_por.append({"Address": df_por.loc[idx, "address"], "Exhausted": True})
    else:
        results_por.append({"Address": df_por.loc[idx, "address"], "Exhausted": False})

# 4. Result Calculation
res_df_por = pd.DataFrame(results_por)
urban_exh = res_df_por[res_df_por["Address"] == "U"]["Exhausted"].mean()
rural_exh = res_df_por[res_df_por["Address"] == "R"]["Exhausted"].mean()

print("✅ PORTUGUESE (E5) REPLICATION COMPLETE!\n")
print(f"Total At-Risk: {len(at_risk_indices)}")
print(f"Urban Exh Rate: {urban_exh:.3f} | Rural Exh Rate: {rural_exh:.3f}")
print(f"ΔExh (Rural - Urban): {rural_exh - urban_exh:.3f}")

E5: Portuguese Dataset Replication shuru holo...

✅ PORTUGUESE (E5) REPLICATION COMPLETE!

Total At-Risk: 228
Urban Exh Rate: 0.434 | Rural Exh Rate: 0.543
ΔExh (Rural - Urban): 0.110
